# Forbidden helicities across decay chains

Decay chains contributing to the same final state can allow different helicities. A component forbidden in one chain must be zero without removing the allowed contribution from another. This matters in both the helicity and canonical-helicity bases.

Consider $\psi(2S)\to\phi\pi^+\pi^-$ through $\phi f_0(980)$ and $\rho(1700)^\pm\pi^\mp$, with $\rho^\pm\to\phi\pi^\pm$.

**Scope.** We use unit dynamics and the default `NoAlignment` to isolate this bookkeeping issue. A physical combined model also requires [spin alignment](https://ampform.readthedocs.io/latest/amplitude/spin-alignment.html) because the spin-1 $\phi$ is produced in different parent rest frames.

<!-- cspell:ignore aslatex clebsch colab emscripten gordan mystnb qrules xreplace -->

In [ ]:
import sys

if sys.platform == "emscripten":
    import piplite

    await piplite.install(["ipywidgets", "qrules==0.10.10", "sympy"])
    await piplite.install("ampform==0.16.2.dev2+gf481d880c")

In [ ]:
import logging

import qrules
import sympy as sp
from ampform import get_builder
from ampform.io import aslatex, improve_latex_rendering
from ampform.sympy import PoolSum, partial_doit
from IPython.display import Markdown, Math
from qrules.settings import NumberOfThreads
from sympy.physics.wigner import clebsch_gordan
from tqdm.auto import tqdm

improve_latex_rendering()
logging.getLogger("qrules.transition").setLevel(logging.ERROR)
NumberOfThreads.set(1)

if sys.platform == "emscripten":
    tqdm.monitor_interval = 0

## Why helicity zero is forbidden

Generate strong-interaction transitions with initial projections $m_\psi=\pm1$. Final-state IDs are $0=\phi$, $1=\pi^+$, and $2=\pi^-$. The pions have helicity zero.

In [ ]:
reaction = qrules.generate_transitions(
    initial_state=("psi(2S)", [-1, 1]),
    final_state=["phi(1020)", "pi+", "pi-"],
    allowed_intermediate_particles=["f(0)(980)", "rho(1700)"],
    allowed_interaction_types="strong",
    formalism="canonical-helicity",
    mass_conservation_factor=0.1,
)
model = get_builder(reaction).formulate()

In [ ]:
helicities_by_resonance = {}
for transition in reaction.transitions:
    resonance = next(iter(transition.intermediate_states.values())).particle.name
    helicities_by_resonance.setdefault(resonance, set()).add(
        transition.final_states[0].spin_projection
    )
rows = ["| Intermediate state | Allowed phi helicities |", "| --- | --- |"]
for resonance, helicities in sorted(helicities_by_resonance.items()):
    values = ", ".join(str(value) for value in sorted(helicities))
    rows.append(f"| {resonance} | {values} |")
Markdown("\n".join(rows))

In the strong decay $\rho(1700)\to\phi\pi$, the spin-parity assignments are $1^-\to1^-\,0^-$. The pion has helicity zero. Parity conservation therefore relates the helicity couplings by

$$
H_{-\lambda_\phi,0}
= P_\rho P_\phi P_\pi\,(-1)^{J_\phi+J_\pi-J_\rho}H_{\lambda_\phi,0}
= -H_{\lambda_\phi,0}.
$$

For $\lambda_\phi=0$, this gives $H_{0,0}=-H_{0,0}$, so **$H_{0,0}=0$**. The transverse couplings can remain nonzero, with $H_{-1,0}=-H_{+1,0}$. Here helicity is defined along the $\phi$ momentum in the $\rho$ rest frame. This restriction belongs to the vector decay; the scalar chain permits a helicity-zero $\phi$.

The same zero appears in the canonical-helicity basis: parity and angular momentum select $L=S=J=1$, and the coefficient $\langle1,0;1,0\mid1,0\rangle$ vanishes. Changing the [coupling basis](https://ampform.readthedocs.io/latest/amplitude/formalism.html) does not remove the need for explicit zero components.

In [ ]:
clebsch_gordan(1, 1, 1, 0, 0, 0)

## Combining the chains

The common sum includes $\lambda_\phi=-1,0,+1$. Superscripts identify the resonant pair: $A^{12}$ is the scalar channel, while $A^{01}$ and $A^{02}$ are the vector channels.

In [ ]:
Math(aslatex(model.intensity))

For $\lambda_\phi=0$, only the scalar channel contributes. AmpForm supplies four zero definitions for the vector channels. They are amplitude components, not extra fit parameters.

In [ ]:
zero_amplitudes = {
    amplitude: expression
    for amplitude, expression in model.amplitudes.items()
    if expression == 0
}
assert len(zero_amplitudes) == 4
Math(aslatex(zero_amplitudes))

## Reproduce the original failure

Before the fix for [AmpForm #529](https://github.com/ComPWA/ampform/issues/529), the builder omitted these zero definitions. Keeping only the nonzero definitions reconstructs that substitution failure with the same intensity and transitions. The four leftover symbols are not fit parameters.

In [ ]:
expanded_intensity = partial_doit(model.intensity, PoolSum, recursive=True)
nonzero_definitions = {a: expr for a, expr in model.amplitudes.items() if expr != 0}
legacy_expression = expanded_intensity.xreplace(nonzero_definitions)
missing = legacy_expression.atoms(sp.Indexed)
assert missing == set(zero_amplitudes)
assert not model.expression.atoms(sp.Indexed)
Math(aslatex({a: a for a in sorted(missing, key=str)}))

## Other channels and both bases

The same pattern occurs in the following selected models. Each contains one scalar channel and both charged vector channels, with initial projections $m=\pm1$. The table is calculated in both bases; each entry gives **total component definitions / explicit zeros**. These examples demonstrate model construction, not the experimental strength of the selected resonances.

In [ ]:
cases = [
    ("psi(2S)", ["phi(1020)", "pi+", "pi-"], "rho(1700)"),
    ("J/psi(1S)", ["omega(782)", "pi+", "pi-"], "rho(1450)"),
    ("J/psi(1S)", ["phi(1020)", "K+", "K-"], "K*(1680)"),
    ("psi(2S)", ["phi(1020)", "K+", "K-"], "K*(1680)"),
]
rows = [
    "| Decay | Resonances | Canonical-helicity | Helicity |",
    "| --- | --- | ---: | ---: |",
]
for initial, final, vector in cases:
    counts = []
    for formalism in ("canonical-helicity", "helicity"):
        selected = qrules.generate_transitions(
            initial_state=(initial, [-1, 1]),
            final_state=final,
            allowed_intermediate_particles=["f(0)(980)", vector],
            allowed_interaction_types="strong",
            formalism=formalism,
            mass_conservation_factor=0.1,
        )
        selected_model = get_builder(selected).formulate()
        zeros = {a for a, expr in selected_model.amplitudes.items() if expr == 0}
        bases = {a.base for a in selected_model.amplitudes}
        assert not any(
            a.base in bases for a in selected_model.expression.atoms(sp.Indexed)
        )
        assert len(selected_model.amplitudes) == 18
        assert zeros == set(zero_amplitudes)
        counts.append(f"{len(selected_model.amplitudes)} / {len(zeros)}")
    decay = f"{initial} → {' '.join(final)}"
    rows.append(f"| {decay} | f(0)(980), {vector} | {' | '.join(counts)} |")
Markdown("\n".join(rows))

## Checking a model

Check for unresolved symbols belonging to the model's amplitude bases. Other indexed objects can be legitimate parameters. With `NoAlignment`, the expanded intensity should also equal the sum of the coherent components built directly from allowed transitions.

In [ ]:
amplitude_bases = {amplitude.base for amplitude in model.amplitudes}
unresolved_amplitudes = {
    symbol
    for symbol in model.expression.atoms(sp.Indexed)
    if symbol.base in amplitude_bases
}
assert not unresolved_amplitudes

direct_intensity = sum(
    expression for name, expression in model.components.items() if name.startswith("I_")
)
assert model.expression == direct_intensity
unresolved_amplitudes

Zero completion covers only the external spin-projection domains of the selected reaction. Alignment may require additional nonzero helicities outside those domains. Do not replace arbitrary unresolved amplitudes with zero; those cases require a complete basis for the alignment method.